# Cloud Forecast Runner Template (Standalone)

This notebook is a fully standalone cloud template for humidity forecasting.

It does not import local project modules. Everything needed for a basic run is defined in notebook cells:
- Runtime/path setup
- Dependency installation
- Optional cloud storage mount
- GRU training and inference with native PyTorch
- Output persistence and artifact export

## 1. Detect Cloud Runtime and Configure Paths

Detect runtime and create cloud-friendly working/data/output directories.
No dependency on local repository structure.

In [ ]:
import os
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
RUNTIME = "colab" if IS_COLAB else "jupyter"

base_root = Path("/content") if IS_COLAB else Path.cwd()
workspace_dir = base_root / "cloud_forecast_workspace"
data_dir = workspace_dir / "data"
output_dir = workspace_dir / "outputs"
checkpoint_dir = workspace_dir / "checkpoints"

for d in [workspace_dir, data_dir, output_dir, checkpoint_dir]:
    d.mkdir(parents=True, exist_ok=True)

print({
    "runtime": RUNTIME,
    "workspace_dir": str(workspace_dir),
    "data_dir": str(data_dir),
    "output_dir": str(output_dir),
    "checkpoint_dir": str(checkpoint_dir),
})

## 2. Install and Verify Dependencies

Install pinned dependencies and validate imports/versions.

In [ ]:
%pip install -q "fastapi>=0.115.0" "uvicorn[standard]>=0.30.0" "torch>=2.3.0" "pandas>=2.2.0" "numpy>=1.26.0" "mysql-connector-python>=9.0.0" "psycopg[binary]>=3.2.0" "python-dotenv>=1.0.1"

In [ ]:
import fastapi
import uvicorn
import torch
import pandas as pd
import numpy as np
import dotenv

print({
    "python": sys.version.split()[0],
    "fastapi": fastapi.__version__,
    "uvicorn": uvicorn.__version__,
    "torch": torch.__version__,
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "python_dotenv": dotenv.__version__ if hasattr(dotenv, '__version__') else 'installed',
})

## 3. Mount/Connect Cloud Storage

Connect optional storage (Google Drive in Colab), and verify access to expected directories.

In [ ]:
# Optional: mount Google Drive for persistent storage in Colab.
if IS_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_root = Path('/content/drive/MyDrive')
        print('Drive mounted:', drive_root)
    except Exception as exc:
        print('Drive mount skipped or failed:', exc)
else:
    drive_root = None

# Storage location for persistent outputs.
persistent_output_dir = (drive_root / 'dadn_outputs' if drive_root else notebook_output_dir)
persistent_output_dir.mkdir(parents=True, exist_ok=True)
print('Persistent output dir:', persistent_output_dir)
print('Sample listing:', [p.name for p in persistent_output_dir.iterdir()][:10])

## 4. Load Configuration and Runtime Parameters

Set dataset source, hyperparameters, and run settings.
If no dataset file is provided, synthetic humidity data is generated automatically.

In [ ]:
from datetime import datetime
import json

# Parameters
history_hours = 72
horizon_hours = 24
lookback = 24
epochs = 20
hidden_size = 32
learning_rate = 1e-3
run_training = True
overwrite_outputs = False

# Optional CSV input. Expected columns: timestamp, humidity
dataset_csv_path = data_dir / "humidity_series.csv"

run_id = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
run_dir = output_dir / run_id
run_dir.mkdir(parents=True, exist_ok=True)

active_config = {
    "history_hours": history_hours,
    "horizon_hours": horizon_hours,
    "lookback": lookback,
    "epochs": epochs,
    "hidden_size": hidden_size,
    "learning_rate": learning_rate,
    "run_training": run_training,
    "overwrite_outputs": overwrite_outputs,
    "dataset_csv_path": str(dataset_csv_path),
    "run_dir": str(run_dir),
}
print(json.dumps(active_config, indent=2))

## 5. Run the Main Processing Pipeline

Load data (CSV or synthetic), preprocess, train a native PyTorch GRU model, and run multi-step forecasting.

In [ ]:
import time
import math
import numpy as np
import pandas as pd
import torch
from torch import nn

start_ts = time.time()

def load_or_generate_data(csv_path: Path, n_points: int = 600) -> pd.DataFrame:
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        if "humidity" not in df.columns:
            raise ValueError("CSV must include 'humidity' column")
        if "timestamp" in df.columns:
            df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
        else:
            df["timestamp"] = pd.date_range("2026-01-01", periods=len(df), freq="h")
        return df[["timestamp", "humidity"]].dropna().reset_index(drop=True)

    # Synthetic hourly humidity signal for standalone cloud run.
    ts = pd.date_range("2026-01-01", periods=n_points, freq="h")
    x = np.arange(n_points)
    humidity = 60 + 8 * np.sin(2 * np.pi * x / 24) + 3 * np.sin(2 * np.pi * x / 24 / 7) + np.random.normal(0, 1.2, n_points)
    humidity = np.clip(humidity, 20, 95)
    return pd.DataFrame({"timestamp": ts, "humidity": humidity})

def make_sequences(values: torch.Tensor, lookback_steps: int):
    if values.numel() <= lookback_steps:
        return None, None
    xs, ys = [], []
    for i in range(lookback_steps, len(values)):
        xs.append(values[i - lookback_steps:i].unsqueeze(-1))
        ys.append(values[i])
    return torch.stack(xs), torch.stack(ys).unsqueeze(-1)

# GRU model built directly from native PyTorch modules.
gru = nn.GRU(input_size=1, hidden_size=hidden_size, num_layers=1, batch_first=True)
head = nn.Linear(hidden_size, 1)

source_df = load_or_generate_data(dataset_csv_path)
series = torch.tensor(source_df["humidity"].astype(float).values, dtype=torch.float32)

if series.numel() < lookback + horizon_hours + 1:
    raise ValueError("Not enough data points for requested lookback and horizon")

# Keep last history+horizon window for report context.
history_window = source_df.tail(history_hours + horizon_hours).reset_index(drop=True)

min_val = float(series.min().item())
max_val = float(series.max().item())
if math.isclose(min_val, max_val):
    raise ValueError("Humidity series has no variance")

normalized = (series - min_val) / (max_val - min_val)
X, y = make_sequences(normalized, lookback)
if X is None:
    raise ValueError("Unable to build training sequences")

train_result = {"status": "skipped"}
if run_training:
    optimizer = torch.optim.Adam(list(gru.parameters()) + list(head.parameters()), lr=learning_rate)
    loss_fn = nn.MSELoss()
    gru.train()
    head.train()
    last_loss = None
    for _ in range(epochs):
        optimizer.zero_grad()
        out, _ = gru(X)
        pred = head(out[:, -1, :])
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        last_loss = float(loss.item())

    checkpoint_path = checkpoint_dir / f"gru_checkpoint_{run_id}.pt"
    torch.save(
        {
            "framework": "pytorch",
            "architecture": "nn.GRU + nn.Linear",
            "gru_state_dict": gru.state_dict(),
            "head_state_dict": head.state_dict(),
            "lookback": lookback,
            "hidden_size": hidden_size,
            "min_val": min_val,
            "max_val": max_val,
            "epochs": epochs,
            "learning_rate": learning_rate,
        },
        checkpoint_path,
    )
    train_result = {
        "status": "trained",
        "loss": last_loss,
        "samples": int(X.size(0)),
        "checkpoint_path": str(checkpoint_path),
    }

# Forecast autoregressively for horizon_hours.
gru.eval()
head.eval()
predictions = []
seed = source_df["humidity"].astype(float).tolist()[-lookback:]
if len(seed) < lookback:
    raise ValueError("Insufficient seed data for forecasting")

window = seed[:]
now = pd.Timestamp.utcnow()
with torch.no_grad():
    for step in range(1, horizon_hours + 1):
        norm = [(v - min_val) / (max_val - min_val) for v in window[-lookback:]]
        x = torch.tensor(norm, dtype=torch.float32).view(1, -1, 1)
        out, _ = gru(x)
        y_hat = float(head(out[:, -1, :]).item())
        value = max(0.0, min(100.0, y_hat * (max_val - min_val) + min_val))
        window.append(value)
        predictions.append(
            {
                "timestamp": (now + pd.Timedelta(hours=step)).isoformat(),
                "value": round(value, 2),
                "lower": round(max(0.0, value - 3.0), 2),
                "upper": round(min(100.0, value + 3.0), 2),
                "confidence": 0.80,
            }
        )

print("Data rows:", len(source_df))
print("Train result:", train_result)
print("Forecast points:", len(predictions))
print("First 3 predictions:", predictions[:3])
print(f"Pipeline elapsed seconds: {time.time() - start_ts:.2f}")

## 6. Write Outputs, Logs, and Checkpoints

Persist predictions, metadata, and logs while avoiding accidental overwrite unless enabled.

In [ ]:
import csv

pred_csv = run_dir / "predictions.csv"
meta_json = run_dir / "run_metadata.json"
log_txt = run_dir / "run.log"

if pred_csv.exists() and not overwrite_outputs:
    pred_csv = run_dir / f"predictions_{run_id}.csv"

with open(pred_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["timestamp", "value", "lower", "upper", "confidence"])
    writer.writeheader()
    for row in predictions:
        writer.writerow(row)

with open(meta_json, "w", encoding="utf-8") as f:
    json.dump(
        {
            "config": active_config,
            "train_result": train_result,
            "prediction_count": len(predictions),
            "history_preview_rows": int(len(history_window)),
        },
        f,
        indent=2,
    )

with open(log_txt, "w", encoding="utf-8") as f:
    f.write(f"run_id={run_id}\n")
    f.write(f"prediction_count={len(predictions)}\n")
    f.write(f"train_status={(train_result or {}).get('status', 'none')}\n")

print("Saved:", pred_csv)
print("Saved:", meta_json)
print("Saved:", log_txt)

## 7. Package and Export Results

Create compressed artifacts and sync/copy to persistent cloud storage.

In [ ]:
import shutil

archive_path = notebook_output_dir / f'forecast_run_{run_id}'
zip_file = shutil.make_archive(str(archive_path), 'zip', root_dir=run_dir)
print('Archive created:', zip_file)

target_zip = persistent_output_dir / Path(zip_file).name
shutil.copy2(zip_file, target_zip)
print('Archive copied to persistent storage:', target_zip)

# Optional: copy model artifact if generated.
model_artifact = ai_root / 'artifacts' / 'models' / 'gru_latest.pt'
if model_artifact.exists():
    shutil.copy2(model_artifact, persistent_output_dir / model_artifact.name)
    print('Model artifact copied:', persistent_output_dir / model_artifact.name)
else:
    print('Model artifact not found (training may have been skipped).')

## Optional: Launch FastAPI in Cloud Runtime

Run the API for manual endpoint testing from the notebook environment.

In [ ]:
# Optional minimal API for cloud endpoint testing (self-contained).
from fastapi import FastAPI
import uvicorn

app = FastAPI(title="Cloud Forecast Demo API")

@app.get("/health")
def health():
    return {"status": "ok", "mode": "standalone-notebook"}

@app.get("/forecast/sample")
def forecast_sample():
    return {
        "horizon_hours": horizon_hours,
        "predictions": predictions[: min(24, len(predictions))],
    }

print("Starting demo API at http://0.0.0.0:8000")
uvicorn.run(app, host="0.0.0.0", port=8000, reload=False)

## Troubleshooting Notes

- If imports fail, rerun dependency install and restart kernel.
- If you want real data, upload a CSV to `data_dir` and set `dataset_csv_path`.
- Required CSV columns: `humidity`; optional: `timestamp`.
- If training is unstable, reduce `learning_rate` or increase `epochs`.
- If API cell blocks execution, interrupt kernel to stop the server.